In [ ]:
import sys, os, io, json, gc
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from PIL import Image
import torchvision.transforms as tvt
from torchvision import datasets as tvd
from scipy.special import softmax, logsumexp
from scipy.optimize import minimize_scalar
from sklearn.metrics import roc_auc_score, roc_curve, f1_score
from tqdm.auto import tqdm
from IPython.display import display

from google.colab import drive
drive.mount('/content/drive')

ROOT = Path("/content/drive/Shareddrives/COS 801 Group 6")
sys.path.insert(0, str(ROOT / "results" / "rq73_inputs"))
import rq73_helpers as h

HO, PRE, CLASSES = h.HANDOFF, h.HANDOFF["preprocessing"], h.CLASSES

# ---------------- config ----------------
MANIFEST_CSV = ROOT / "data" / "data_split_resolved.csv"                 # same path as before; only used to find the held-out PlantVillage folders
CASSAVA_DIR = ROOT / "data/raw/cassava-leaf-disease-classification/train_images"
PV_COLOR_DIR = ROOT / "data/raw/plantvillage/plantvillage dataset/color"
CASSAVA_N, HELD_OUT_PER_FOLDER, NONPLANT_N = 5000, 300, 2000             # lower CASSAVA_N if Colab runs out of RAM
ACCEPT_RATE = 0.95            # threshold accepts this share of lab_val images
PRIMARY = "energy"            # decided in advance, not picked from the results
CRITERION = 0.10              # placeholder for "reliable": at most this share of OOD images accepted. Set your own.
SEED = 42
OUT = ROOT / "results" / "rq73_outputs"; OUT.mkdir(parents=True, exist_ok=True)
CACHE = OUT / "cache"; CACHE.mkdir(exist_ok=True)                        # logits, kept on Drive
PIXDIR = Path("/content/rq73_pix"); PIXDIR.mkdir(exist_ok=True)          # preprocessed pixels, kept local (rebuilt if the runtime resets)

RUN_KEYS = [m["key"] for m in HO["models"] if m["arch"] == "mobilenet_v3_small"]   # all 13 MobileNetV3 models
# first pass with the baseline only (faster): uncomment the next line
# RUN_KEYS = ["baseline_mobilenet_v3_small"]
# -----------------------------------------

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
META = {m["key"]: m for m in HO["models"]}
print("device:", DEVICE, "|", len(RUN_KEYS), "models:", RUN_KEYS)

# the saved in-distribution logits; check that the split names match the handoff rules
z0 = h.load_logits(RUN_KEYS[0], root=str(ROOT))
print({k: z0[k].shape for k in z0})
assert all(f"logits_{s}" in z0 for s in HO["splits"]), f"split names differ from handoff, saved keys: {list(z0)}"

Mounted at /content/drive
device: cpu | 13 models: ['A0_none__seed42', 'A0_none__seed43', 'A0_none__seed44', 'A4_combined__seed42', 'A4_combined__seed43', 'A4_combined__seed44', 'baseline_mobilenet_v3_small', 'global_magnitude40-rec__A0_none__seed42', 'global_magnitude40-rec__A0_none__seed43', 'global_magnitude40-rec__A0_none__seed44', 'global_magnitude40-rec__A4_combined__seed42', 'global_magnitude40-rec__A4_combined__seed43', 'global_magnitude40-rec__A4_combined__seed44']
{'classes': (26,), 'logits_lab_val': (5385, 26), 'y_lab_val': (5385,), 'path_lab_val': (5385,), 'logits_lab_test': (5376, 26), 'y_lab_test': (5376,), 'path_lab_test': (5376,), 'logits_field_test': (2499, 26), 'y_field_test': (2499,), 'path_field_test': (2499,)}


In [ ]:
RS = tvt.Resize(tuple(PRE["resize"]))

def pil_like_helper(im):
    """For images that aren't files (CIFAR/DTD): the rule the helper applies to non-JPEG input."""
    im = im.convert("RGB")
    if max(im.size) > PRE["cache_max_side"]:
        im.thumbnail((PRE["cache_max_side"],) * 2, Image.BILINEAR)
    buf = io.BytesIO(); im.save(buf, "JPEG", quality=PRE["jpeg_quality"]); buf.seek(0)
    return Image.open(buf).convert("RGB")

def file_pixels(path):                       # helper's load_image, then the helper's resize. uint8 is exact (ToTensor only divides)
    return np.asarray(RS(h.load_image(path)), dtype=np.uint8)

def build_cache(name, items, fn=file_pixels, workers=24):
    f = PIXDIR / f"{name}_n{len(items)}.npy"
    if f.exists(): return np.load(f)
    with ThreadPoolExecutor(workers) as ex:
        arr = np.stack(list(tqdm(ex.map(fn, items), total=len(items), desc=name)))
    np.save(f, arr); return arr

# ---- in-distribution sets: the group's own splits, by the paths saved with the logits ----
for k in RUN_KEYS:                           # every model must use the same images
    zk = h.load_logits(k, root=str(ROOT))
    assert all(np.array_equal(zk[f"path_{s}"], z0[f"path_{s}"]) for s in HO["splits"]), f"{k} uses different images"
ID_PIX = {s: build_cache(f"id_{s}", [ROOT / str(p) for p in z0[f"path_{s}"]]) for s in HO["splits"]}

# ---- OOD set 1: Cassava (random subset) ----
rng = np.random.default_rng(SEED)
cass = sorted(CASSAVA_DIR.glob("*.jpg"))
cass = [cass[i] for i in sorted(rng.choice(len(cass), size=min(CASSAVA_N, len(cass)), replace=False))]
OOD_PIX = {"cassava": build_cache("cassava", cass)}

# ---- OOD sets 2-4: the PlantVillage classes the models never trained on ----
man = pd.read_csv(MANIFEST_CSV, sep=None, engine="python")
trained_folders = {Path(p).parent.name for p in man.loc[man["domain"] == "lab", "path"]}
trained_crops = {f.split("___")[0] for f in trained_folders}
held_out = sorted(d.name for d in PV_COLOR_DIR.iterdir() if d.is_dir() and d.name not in trained_folders)
print(f"{len(held_out)} held-out PlantVillage folders:", held_out)
groups = {"pv_novel_crop": [], "pv_unseen_disease_known_crop": [], "pv_unseen_healthy_known_crop": []}
for f in held_out:
    imgs = sorted((PV_COLOR_DIR / f).glob("*.[jJ][pP][gG]"))
    imgs = [imgs[i] for i in rng.choice(len(imgs), size=min(HELD_OUT_PER_FOLDER, len(imgs)), replace=False)]
    grp = ("pv_novel_crop" if f.split("___")[0] not in trained_crops
           else "pv_unseen_healthy_known_crop" if f.lower().endswith("healthy") else "pv_unseen_disease_known_crop")
    groups[grp] += imgs
for n, paths in groups.items():
    if paths: OOD_PIX[n] = build_cache(n, paths)

# ---- OOD sets 5-6: non-plant images ----
cifar = tvd.CIFAR10("/content/cifar", train=False, download=True)
OOD_PIX["nonplant_cifar10"] = build_cache("cifar10", list(range(NONPLANT_N)),
                                          lambda i: np.asarray(RS(pil_like_helper(cifar[i][0])), dtype=np.uint8))
try:
    dtd = tvd.DTD("/content/dtd", split="test", download=True)
    OOD_PIX["nonplant_textures_dtd"] = build_cache("dtd", list(range(min(NONPLANT_N, len(dtd)))),
                                                   lambda i: np.asarray(RS(pil_like_helper(dtd[i][0])), dtype=np.uint8))
except Exception as e:
    print("DTD skipped:", e)

print("ID :", {k: v.shape[0] for k, v in ID_PIX.items()})
print("OOD:", {k: v.shape[0] for k, v in OOD_PIX.items()})

id_lab_val:   0%|          | 0/5385 [00:00<?, ?it/s]

id_lab_test:   0%|          | 0/5376 [00:00<?, ?it/s]

In [ ]:
MEAN = torch.tensor(PRE["mean"]).view(1, 3, 1, 1); STD = torch.tensor(PRE["std"]).view(1, 3, 1, 1)
SCORES = ["msp", "msp_temp", "max_logit", "energy"]
ORDER_OOD = [n for n in ["cassava", "pv_novel_crop", "pv_unseen_disease_known_crop",
                         "pv_unseen_healthy_known_crop", "nonplant_cifar10", "nonplant_textures_dtd"] if n in OOD_PIX]

@torch.no_grad()
def logits_from_pix(net, pix, bs=128):
    out = []
    for i in range(0, len(pix), bs):
        x = torch.from_numpy(pix[i:i + bs]).permute(0, 3, 1, 2).float().div_(255)
        out.append(net(((x - MEAN) / STD).to(DEVICE)).float().cpu().numpy())
    return np.concatenate(out)

def model_logits(key):
    """OOD logits for one model, cached on Drive."""
    f = CACHE / f"logits_{key}.npz"
    if f.exists():
        d = np.load(f); return {k: d[k] for k in d.files}
    net = h.load_model(key, root=str(ROOT)).to(DEVICE).eval()
    out = {f"ood__{n}": logits_from_pix(net, p) for n, p in OOD_PIX.items()}
    np.savez(f, **out); del net; gc.collect()
    return out

def fit_T(lv, yv):
    lv64 = lv.astype(np.float64)
    def nll(lt):
        zz = lv64 / np.exp(lt)
        return -(zz[np.arange(len(zz)), yv] - logsumexp(zz, axis=1)).mean()
    return float(np.exp(minimize_scalar(nll, bounds=(-3, 3), method="bounded").x))

def scores(lg, T=1.0):
    lg = lg.astype(np.float64)                       # float64 avoids exact-1.0 ties
    return {"msp": softmax(lg, 1).max(1), "msp_temp": softmax(lg / T, 1).max(1),
            "max_logit": lg.max(1), "energy": logsumexp(lg, axis=1)}

def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    w = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - w, c + w

def aurc(conf, correct):
    order = np.argsort(-conf)
    return (1 - np.cumsum(correct[order]) / np.arange(1, len(conf) + 1)).mean()

def ece(lg, y, T=1.0, n_bins=15):
    p = softmax(lg.astype(np.float64) / T, axis=1); conf, ok = p.max(1), p.argmax(1) == y
    bins = np.linspace(0, 1, n_bins + 1); tot = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): tot += m.mean() * abs(ok[m].mean() - conf[m].mean())
    return tot

def evaluate_logits(name, lv, yv, lt, yt, lf, yf, ood):
    T = fit_T(lv, yv)                                # fitted on lab_val only
    sv, st, sf = scores(lv, T), scores(lt, T), scores(lf, T)
    so = {n: scores(l, T) for n, l in ood.items()}
    okt, okf = lt.argmax(1) == yt, lf.argmax(1) == yf
    id_rows, ood_rows = [], []
    for s in SCORES:
        tau = np.percentile(sv[s], (1 - ACCEPT_RATE) * 100)       # lab_val only
        at, af = st[s] >= tau, sf[s] >= tau
        id_rows.append({"key": name, "score": s, "T": T, "tau": tau,
            "lab_acc": okt.mean(), "lab_coverage": at.mean(), "lab_selective_acc": okt[at].mean(),
            "lab_aurc": aurc(st[s], okt), "lab_ece": ece(lt, yt), "lab_ece_temp": ece(lt, yt, T),
            "field_acc": okf.mean(), "field_coverage": af.mean(),
            "field_selective_acc": okf[af].mean() if af.any() else np.nan,
            "field_aurc": aurc(sf[s], okf), "field_ece": ece(lf, yf), "field_ece_temp": ece(lf, yf, T)})
        for n, d in so.items():
            acc = d[s] >= tau; k, m = int(acc.sum()), len(acc); lo, hi = wilson(k, m)
            ood_rows.append({"key": name, "score": s, "ood_set": n, "n": m, "false_accept": k / m,
                "fa_lo": lo, "fa_hi": hi,
                "auroc_vs_lab": roc_auc_score(np.r_[np.ones(len(st[s])), np.zeros(m)], np.r_[st[s], d[s]]),
                "auroc_vs_field": roc_auc_score(np.r_[np.ones(len(sf[s])), np.zeros(m)], np.r_[sf[s], d[s]])})
    return id_rows, ood_rows

def evaluate_model(key):
    zk, ood = h.load_logits(key, root=str(ROOT)), model_logits(key)
    return evaluate_logits(key, zk["logits_lab_val"], zk["y_lab_val"], zk["logits_lab_test"], zk["y_lab_test"],
                           zk["logits_field_test"], zk["y_field_test"],
                           {n: ood[f"ood__{n}"] for n in OOD_PIX})

# ---- check 1: my pixels + the helper's model reproduce the group's saved logits ----
net = h.load_model(RUN_KEYS[0], root=str(ROOT)).to(DEVICE).eval()
for s in ["lab_val", "field_test"]:
    d = np.abs(logits_from_pix(net, ID_PIX[s][:256]) - z0[f"logits_{s}"][:256]).max()
    print(f"{RUN_KEYS[0]}, {s}: max |difference| from the saved logits = {d:.5f}   (expect about 1e-3 or less)")
del net

# ---- check 2: the saved logits agree with the macro F1 the group reported ----
for k in RUN_KEYS:
    zk = h.load_logits(k, root=str(ROOT))
    f1 = f1_score(zk["y_lab_test"], zk["logits_lab_test"].argmax(1), average="macro")
    flag = "" if abs(f1 - META[k]["lab_test_macro_f1"]) < 0.002 else "   <-- CHECK"
    print(f"{k:50s} lab_test macro F1 {f1:.4f} vs reported {META[k]['lab_test_macro_f1']:.4f}{flag}")

In [ ]:
all_id, all_ood = [], []
for key in tqdm(RUN_KEYS):
    a, b = evaluate_model(key)
    all_id += a; all_ood += b

ID = pd.DataFrame(all_id); OODT = pd.DataFrame(all_ood)
for D in (ID, OODT):
    D["arm"] = D["key"].map(lambda k: META[k]["arm"]); D["recipe"] = D["key"].map(lambda k: META[k]["recipe"])
    D["seed"] = D["key"].map(lambda k: META[k]["seed"])
ID.to_csv(OUT / "abstention_id_all_models.csv", index=False)
OODT.to_csv(OUT / "abstention_ood_all_models.csv", index=False)
print("done:", len(RUN_KEYS), "models")

In [ ]:
import platform, datetime
ROWS = ["arm", "recipe"]

def ms(df, value, rows, cols):
    g = df.groupby(rows + [cols])[value].agg(["mean", "std", "count"]).reset_index()
    g["txt"] = [f"{m:.3f}" + (f" ± {s:.3f}" if c > 1 else "") for m, s, c in zip(g["mean"], g["std"], g["count"])]
    return g.pivot(index=rows, columns=cols, values="txt")

P = ID[ID["score"] == PRIMARY]
print(f"--- known classes, '{PRIMARY}' threshold fitted on lab_val to accept {ACCEPT_RATE:.0%} (lab_test and PlantDoc are report-only) ---")
cols = ["lab_acc", "lab_coverage", "lab_ece", "lab_ece_temp", "field_acc", "field_coverage", "field_selective_acc", "field_ece"]
display(P.groupby(ROWS)[cols].mean().style.format("{:.3f}"))

PO = OODT[OODT["score"] == PRIMARY]
print(f"\n--- false-accept rate on unfamiliar inputs, '{PRIMARY}' (lower = better; mean ± std over seeds) ---")
display(ms(PO, "false_accept", ROWS, "ood_set")[ORDER_OOD])
print("\n--- AUROC: lab_test (known, lab photos) vs each OOD set ---")
display(ms(PO, "auroc_vs_lab", ROWS, "ood_set")[ORDER_OOD])
print("\n--- AUROC: PlantDoc (known crops, field photos) vs each OOD set. This removes the lab-vs-field difference for Cassava and non-plant. ---")
display(ms(PO, "auroc_vs_field", ROWS, "ood_set")[ORDER_OOD])

print(f"\n--- RQ7.3 criterion: false-accept at most {CRITERION:.0%}. Seeds meeting it, out of the seeds run ---")
v = PO.assign(ok=PO["false_accept"] <= CRITERION).groupby(ROWS + ["ood_set"])["ok"].agg(["sum", "count"]).reset_index()
v["txt"] = v["sum"].astype(int).astype(str) + "/" + v["count"].astype(str)
display(v.pivot(index=ROWS, columns="ood_set", values="txt")[ORDER_OOD])

dep = [k for k in HO["deployment"]["keys"] if k in RUN_KEYS]
if dep:
    print("\n--- all four scores, deployment model only ---")
    display(ms(OODT[OODT["key"].isin(dep)], "false_accept", ["score"], "ood_set")[ORDER_OOD])

# settings of this run (section 4.5)
with open(OUT / "run_settings.json", "w") as f:
    json.dump({"run_time": datetime.datetime.now().isoformat(timespec="seconds"), "device": str(DEVICE),
               "torch": torch.__version__, "python": platform.python_version(), "seed": SEED,
               "accept_rate": ACCEPT_RATE, "primary_score": PRIMARY, "criterion": CRITERION,
               "cassava_n": CASSAVA_N, "held_out_per_folder": HELD_OUT_PER_FOLDER, "nonplant_n": NONPLANT_N,
               "keys": RUN_KEYS, "ood_sizes": {k: int(len(v)) for k, v in OOD_PIX.items()},
               "id_sizes": {k: int(len(v)) for k, v in ID_PIX.items()}, "preprocessing": PRE}, f, indent=2, default=str)
print("\nsaved tables and settings to", OUT)

In [ ]:
def plot_model(key, score=PRIMARY):
    zk, ood = h.load_logits(key, root=str(ROOT)), model_logits(key)
    T = fit_T(zk["logits_lab_val"], zk["y_lab_val"])
    tau = ID.query("key == @key and score == @score")["tau"].iloc[0]
    sc = lambda lg: scores(lg, T)[score]
    sets = {"lab_test (known, lab)": sc(zk["logits_lab_test"]), "PlantDoc (known, field)": sc(zk["logits_field_test"])}
    for n in ["cassava", "pv_unseen_disease_known_crop", "nonplant_cifar10"]:
        if n in OOD_PIX: sets[n] = sc(ood[f"ood__{n}"])

    fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
    for n, vals in sets.items(): ax[0].hist(vals, bins=50, alpha=0.5, density=True, label=n)
    ax[0].axvline(tau, color="k", ls="--", label="threshold (lab_val)")
    ax[0].set(title=f"{score} distributions", xlabel=score); ax[0].legend(fontsize=7)

    for nm, lg, y in [("lab_test", zk["logits_lab_test"], zk["y_lab_test"]),
                      ("PlantDoc (field)", zk["logits_field_test"], zk["y_field_test"])]:
        s = sc(lg); ok = lg.argmax(1) == y; o = np.argsort(-s); k = np.arange(1, len(s) + 1)
        ax[1].plot(k / len(s), 1 - np.cumsum(ok[o]) / k, label=nm)
    ax[1].set(xlabel="coverage", ylabel="error among accepted", title="Risk-coverage"); ax[1].legend()

    if "cassava" in sets:
        for nm, ref in [("lab_test vs cassava", sets["lab_test (known, lab)"]), ("PlantDoc vs cassava", sets["PlantDoc (known, field)"])]:
            y = np.r_[np.ones(len(ref)), np.zeros(len(sets["cassava"]))]
            fpr, tpr, _ = roc_curve(y, np.r_[ref, sets["cassava"]]); ax[2].plot(fpr, tpr, label=nm)
    ax[2].plot([0, 1], [0, 1], "k:"); ax[2].set(xlabel="cassava accepted", ylabel="known accepted", title="ROC"); ax[2].legend()
    fig.suptitle(key); fig.tight_layout()
    fig.savefig(OUT / f"abstention_plots_{key}.png", dpi=200, bbox_inches="tight"); plt.show()

if "baseline_mobilenet_v3_small" in RUN_KEYS: plot_model("baseline_mobilenet_v3_small")
if dep: plot_model(dep[0])

In [ ]:
!pip install -q onnxruntime
import onnxruntime as ort

SEED_ONNX, SUB = 42, 1500                       # SUB = images per set; lower it to 800 if this is slow
dep_key = f"global_magnitude40-rec__A4_combined__seed{SEED_ONNX}"
zk, fl_ood = h.load_logits(dep_key, root=str(ROOT)), model_logits(dep_key)

PIX_ALL = {**{f"id__{s}": p for s, p in ID_PIX.items()}, **{f"ood__{n}": p for n, p in OOD_PIX.items()}}
rs = np.random.default_rng(0)
IDX = {n: np.sort(rs.choice(len(p), size=min(SUB, len(p)), replace=False)) for n, p in PIX_ALL.items()}

def onnx_logits(path, pix, bs=64):
    sess = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
    inp = sess.get_inputs()[0]
    if isinstance(inp.shape[0], int): bs = inp.shape[0]          # some exports have a fixed batch size
    out = []
    for i in range(0, len(pix), bs):
        x = torch.from_numpy(pix[i:i + bs]).permute(0, 3, 1, 2).float().div_(255)
        out.append(sess.run(None, {inp.name: ((x - MEAN) / STD).numpy()})[0])
    return np.concatenate(out)

def eval_on_subsets(name, get):                    # get(set_name) -> logits on the subset
    L = {n: get(n) for n in PIX_ALL}
    y = lambda s: zk[f"y_{s}"][IDX[f"id__{s}"]]
    return evaluate_logits(name, L["id__lab_val"], y("lab_val"), L["id__lab_test"], y("lab_test"),
                           L["id__field_test"], y("field_test"), {n: L[f"ood__{n}"] for n in OOD_PIX})

# the float model on exactly the same subsets, from the saved/cached logits
float_get = lambda n: (zk[f"logits_{n[4:]}"] if n.startswith("id__") else fl_ood[n])[IDX[n]]
rows_id, rows_ood = [], []
a, b = eval_on_subsets("float (pytorch)", float_get); rows_id += a; rows_ood += b

files = [p for p in HO["deployment"]["exported_files"] if p.endswith(".onnx") and f"seed{SEED_ONNX}/" in p]
for f in files:
    variant = Path(f).stem.replace("model_", "onnx_")
    try:
        get = lambda n, f=f: onnx_logits(ROOT / f, PIX_ALL[n][IDX[n]])
        a, b = eval_on_subsets(variant, get); rows_id += a; rows_ood += b
        print("done", variant)
    except Exception as e:
        print(f"skipping {variant}: {e!r}")

qid, qood = pd.DataFrame(rows_id), pd.DataFrame(rows_ood)
qid, qood = qid[qid["score"] == PRIMARY].set_index("key"), qood[qood["score"] == PRIMARY]
print(f"--- seed {SEED_ONNX}, '{PRIMARY}': does quantisation change abstention? (same {SUB}-image subsets for every row) ---")
display(qid[["lab_acc", "lab_coverage", "lab_ece", "field_acc", "field_coverage", "field_selective_acc"]].style.format("{:.3f}"))
display(qood.pivot(index="key", columns="ood_set", values="false_accept")[ORDER_OOD].style.format("{:.3f}"))
qid.to_csv(OUT / f"quantisation_check_id_seed{SEED_ONNX}.csv"); qood.to_csv(OUT / f"quantisation_check_ood_seed{SEED_ONNX}.csv", index=False)